In [2]:
library(tidyverse)
library(phyloseq)
library(readxl)

In [3]:
extraction_batches_raw <- readxl::read_xlsx("Extraction Batches.xlsx")

New names:
• `` -> `...5`


In [60]:
extraction_batches <- dplyr::mutate(extraction_batches_raw,
                              sequenced_tag = if_else(lab_blank %in% Not_sequenced, "Not_sequenced", "Sequenced"),
                              lab_blank = if_else(lab_blank %in% Not_sequenced, NA_character_, lab_blank),
                              date = as.Date(date, format = "%d/%m/%Y"),
                              #Set date as NA if thata is not sequenced to ensure correct batch assignment
                              assigned_date = if_else(sequenced_tag == "Not_sequenced", NA, date)) |>
                              #This assigns 
                              tidyr::fill(assigned_date, lab_blank) |>
                       dplyr::group_by(assigned_date, lab_blank, field_blank) |>
                       dplyr::mutate(batch = paste0("Batch_", dplyr::cur_group_id())) |>
                       dplyr::ungroup() |>
                       dplyr::mutate(batch = factor(batch, levels = sort(unique(batch)))) |>
                       dplyr::select(batch, date, assigned_date, sample_id, lab_blank, field_blank, sequenced_tag)

In [61]:
extraction_batches

batch,date,assigned_date,sample_id,lab_blank,field_blank,sequenced_tag
<fct>,<date>,<date>,<chr>,<chr>,<chr>,<chr>
Batch_1,2026-05-07,2026-05-07,T1W1,T1L1,T1WB,Sequenced
Batch_1,2026-05-07,2026-05-07,T1W2,T1L1,T1WB,Sequenced
Batch_1,2026-05-07,2026-05-07,T1W3,T1L1,T1WB,Sequenced
Batch_2,2026-05-07,2026-05-07,T1SG1,T1SGL1,NA,Sequenced
Batch_2,2026-05-07,2026-05-07,T1SG2,T1SGL1,NA,Sequenced
Batch_2,2026-05-07,2026-05-07,T1SG3,T1SGL1,NA,Sequenced
Batch_2,2026-05-08,2026-05-07,T2SG1,T1SGL1,NA,Not_sequenced
Batch_2,2026-05-08,2026-05-07,T2SG2,T1SGL1,NA,Not_sequenced
Batch_2,2026-05-08,2026-05-07,T2SG3,T1SGL1,NA,Not_sequenced


In [6]:
(phyloseq_16S <- read_rds("16S_phyloseq.rds"))

phyloseq-class experiment-level object
otu_table()   OTU Table:         [ 17269 taxa and 53 samples ]
sample_data() Sample Data:       [ 53 samples by 7 sample variables ]
tax_table()   Taxonomy Table:    [ 17269 taxa by 7 taxonomic ranks ]
phy_tree()    Phylogenetic Tree: [ 17269 tips and 17253 internal nodes ]

In [46]:
sample_data(phyloseq_16S) |> head(3)

,transect,habitat,microbial_community,sample,plant_species_1,plant_species_2,blank_group
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
RemovePrimer_Final.T1IS1_8522605005578,T1,intertidal_saltmarsh,prokaryote,T1IS1,Salicornia_tegataria,Triglochin_striata,Real Sample
RemovePrimer_Final.T1IS2_8522605005579,T1,intertidal_saltmarsh,prokaryote,T1IS2,Salicornia_tegataria,Triglochin_striata,Real Sample
RemovePrimer_Final.T1IS3_8522605005580,T1,intertidal_saltmarsh,prokaryote,T1IS3,Salicornia_tegataria,Triglochin_striata,Real Sample


In [34]:
temp_sample_data <- data.frame(sample_data(phyloseq_16S))
temp_sample_data$Read_tag <- row.names(temp_sample_data)

In [47]:
head(temp_sample_data, 3)

,transect,habitat,microbial_community,sample,plant_species_1,plant_species_2,blank_group,Read_tag
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
RemovePrimer_Final.T1IS1_8522605005578,T1,intertidal_saltmarsh,prokaryote,T1IS1,Salicornia_tegataria,Triglochin_striata,Real Sample,RemovePrimer_Final.T1IS1_8522605005578
RemovePrimer_Final.T1IS2_8522605005579,T1,intertidal_saltmarsh,prokaryote,T1IS2,Salicornia_tegataria,Triglochin_striata,Real Sample,RemovePrimer_Final.T1IS2_8522605005579
RemovePrimer_Final.T1IS3_8522605005580,T1,intertidal_saltmarsh,prokaryote,T1IS3,Salicornia_tegataria,Triglochin_striata,Real Sample,RemovePrimer_Final.T1IS3_8522605005580


In [48]:
head(extraction_batches, 3)

batch,date,assigned_date,sample_id,lab_blank,field_blank,sequenced_tag
<chr>,<date>,<date>,<chr>,<chr>,<chr>,<chr>
Batch_1,2026-05-07,2026-05-07,T1W1,T1L1,T1WB,Sequenced
Batch_1,2026-05-07,2026-05-07,T1W2,T1L1,T1WB,Sequenced
Batch_1,2026-05-07,2026-05-07,T1W3,T1L1,T1WB,Sequenced


In [51]:
extraction_batches_all <- merge(temp_sample_data, extraction_batches, by.x = "sample", by.y = "sample_id")

In [53]:
head(extraction_batches_all, 3)

,sample,transect,habitat,microbial_community,plant_species_1,plant_species_2,blank_group,Read_tag,batch,date,assigned_date,lab_blank,field_blank,sequenced_tag
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<date>,<date>,<chr>,<chr>,<chr>
1,T1IS1,T1,intertidal_saltmarsh,prokaryote,Salicornia_tegataria,Triglochin_striata,Real Sample,RemovePrimer_Final.T1IS1_8522605005578,Batch_3,2026-05-08,2026-05-08,T1ISL1,NA,Sequenced
2,T1IS2,T1,intertidal_saltmarsh,prokaryote,Salicornia_tegataria,Triglochin_striata,Real Sample,RemovePrimer_Final.T1IS2_8522605005579,Batch_3,2026-05-08,2026-05-08,T1ISL1,NA,Sequenced
3,T1IS3,T1,intertidal_saltmarsh,prokaryote,Salicornia_tegataria,Triglochin_striata,Real Sample,RemovePrimer_Final.T1IS3_8522605005580,Batch_3,2026-05-08,2026-05-08,T1ISL1,NA,Sequenced


In [19]:
# tibble::rownames_to_column(var = "Read_tag") |>
# data.frame() |>
# dplyr::mutate(tmp_sample = sample) |>
# tibble::column_to_rownames(var = "tmp_sample")     

In [57]:
#head(temp_sample_data)

In [56]:
#head(extraction_batches)

In [39]:
#sample_data(phyloseq_16S) <- temp_sample_data

In [42]:
#head(temp_sample_data)

In [63]:
lapply(unique(extraction_batches$batch), function(current_batch) {
    
      extraction_batch <- extraction_batches |> dplyr::filter(batch == current_batch)
      physeq_subset <- subset_samples(physeq, sample_id %in% samples_to_keep)
        
})

batch,date,assigned_date,sample_id,lab_blank,field_blank,sequenced_tag
<fct>,<date>,<date>,<chr>,<chr>,<chr>,<chr>
Batch_1,2026-05-07,2026-05-07,T1W1,T1L1,T1WB,Sequenced
Batch_1,2026-05-07,2026-05-07,T1W2,T1L1,T1WB,Sequenced
Batch_1,2026-05-07,2026-05-07,T1W3,T1L1,T1WB,Sequenced
batch,date,assigned_date,sample_id,lab_blank,field_blank,sequenced_tag
<fct>,<date>,<date>,<chr>,<chr>,<chr>,<chr>
Batch_2,2026-05-07,2026-05-07,T1SG1,T1SGL1,NA,Sequenced
Batch_2,2026-05-07,2026-05-07,T1SG2,T1SGL1,NA,Sequenced
Batch_2,2026-05-07,2026-05-07,T1SG3,T1SGL1,NA,Sequenced
Batch_2,2026-05-08,2026-05-07,T2SG1,T1SGL1,NA,Not_sequenced
